In [1]:
import os
import time

import tools
import benchmarks as bm
from poseEvaluation import PoseErrorEvaluator
from evo.core import metrics

[DEBUG][2024-02-08 00:26:56,771][log.configure_logging():112]
System info:
Python 3.8.17
Linux-6.5.0-15-generic-x86_64-with-glibc2.17
slam@slam-TensorBook



In [2]:
benchmark_list = ["SenseTime","TUMRGBD1", "TUMRGBD2", "TUMRGBD3"]
#benchmark_list = ["SenseTime", "TUMRGBD1", "TUMRGBD2", "TUMRGBD3"]
benchmark_list = ["SenseTime"]
root_dir = "/home/slam/Projects/ORB_SLAM3_DRL/"
trials = 3
delta = 30
max_diff = 0.05
max_null_length = 10

os.chdir(root_dir)
root_dir = "."

In [3]:
poseErrorEvaluator = PoseErrorEvaluator(root_dir, delta, max_diff, max_null_length)

In [4]:
tools.delete_files_in_folder("./logs")
tools.delete_files_in_folder("./data")
tools.delete_files_in_folder("./figures")

Deleted: ./logs/ground_truth.txt
Deleted: ./logs/trajectory.txt
All files deleted successfully.
All files deleted successfully.
All files deleted successfully.


In [5]:
for i, benchmark in enumerate(benchmark_list):
    benchmarkClass = bm.benchmark_factory[benchmark]
    benchmarkObject = benchmarkClass(root_dir)
    benchmarkObject.generate_script()
    scriptList = benchmarkObject.get_script()
    for j, scriptDict in enumerate(scriptList):    
        time.sleep(1)
        benchmark = scriptDict["benchmark"]
        trajectory = scriptDict["trajectory"]
        benchmarkObject.copy_ground_truth_traj(scriptDict["trajectory"])
        
        for trial in range(0, trials):
            # Baseline
            baseline_df, baseline_df_traj, baseline_initial_rows = benchmarkObject.load_n_save_baseline_estimated_traj(trajectory, trial)
            time.sleep(1)

            poseErrorEvaluator = PoseErrorEvaluator(root_dir, delta, max_diff, max_null_length)

            poseErrorEvaluator.load_trajectory()
            traj_est, traj_ref = poseErrorEvaluator.get_traj_w_gt()

            rpe_metric = poseErrorEvaluator.calculate_RE()
            poseErrorEvaluator.merge_feature_with_label(trajectory)
            merged_df = poseErrorEvaluator.get_feature_w_label()
            error_df = poseErrorEvaluator.get_error_df()


            poseErrorEvaluator.plot_aligned_trajectory(traj_est, traj_ref, benchmark, trajectory, trial, "Baseline")
            poseErrorEvaluator.plot_error(error_df, delta, benchmark, trajectory, trial, "Baseline")

            poseErrorEvaluator.plot_trajectory_with_error(rpe_metric, traj_ref, traj_est,
                                                          benchmark, trajectory, trial, "Baseline")


            #df_data = poseErrorEvaluator.attach_label_2_features(delta, error, df)
            #time.sleep(1)

            merged_df.to_csv("./data/baseline_{}_{}_{}.csv".format(benchmark,trajectory, trial), sep=',', index=False, header=True)
            time.sleep(2)
            
            
            # RL enabled
            rl_df, rl_df_traj, rl_initial_rows = benchmarkObject.load_n_save_rl_estimated_traj(trajectory, trial)
            time.sleep(1)
            
            poseErrorEvaluator = PoseErrorEvaluator(root_dir, delta, max_diff, max_null_length)
            
            poseErrorEvaluator.load_trajectory()
            traj_est, traj_ref = poseErrorEvaluator.get_traj_w_gt()
            
            rpe_metric = poseErrorEvaluator.calculate_RE(pose_relation=metrics.PoseRelation.point_distance)
            
            poseErrorEvaluator.merge_feature_with_label(trajectory, trial)
            merged_df = poseErrorEvaluator.get_feature_w_label()
            error_df = poseErrorEvaluator.get_error_df()
            
        
            poseErrorEvaluator.plot_aligned_trajectory(traj_est, traj_ref, benchmark, trajectory, trial, "RL")
            poseErrorEvaluator.plot_error(error_df, delta, benchmark, trajectory, trial, "RL")
            
            poseErrorEvaluator.plot_trajectory_with_error(rpe_metric, traj_ref, traj_est,
                                                          benchmark, trajectory, trial, "RL")

            merged_df.to_csv("./data/RL_{}_{}_{}.csv".format(benchmark,trajectory,trial), sep=',', index=False, header=True)
            time.sleep(1)
            
            reward_df = poseErrorEvaluator.calculate_reward(trajectory, trial)
            reward_df.to_csv("./RL_Train/RL_{}_{}_{}.csv".format(benchmark,trajectory,trial), sep=',', index=False, header=True)
            

[DEBUG][2024-02-08 00:27:02,380][file_interface.read_tum_trajectory_file():117]
Loaded 1554 stamps and poses from: ./logs/trajectory.txt
[DEBUG][2024-02-08 00:27:02,449][file_interface.read_tum_trajectory_file():117]
Loaded 24578 stamps and poses from: ./logs/ground_truth.txt
Subtrajectories for alignment: [[0, 1005], [1005, 1554]]
Original estimated subtrajectory length: 1005
[DEBUG][2024-02-08 00:27:02,478][sync.associate_trajectories():110]
Found 1005 of max. 1005 possible matching timestamps between...
	first trajectory
and:	second trajectory
..with max. time diff.: 0.05 (s) and time offset: 0.0 (s).
[DEBUG][2024-02-08 00:27:02,479][trajectory.align():202]
Aligning using Umeyama's method... (with scale correction)
[DEBUG][2024-02-08 00:27:02,538][trajectory.align():214]
Rotation of alignment:
[[-0.14525576  0.98111172  0.12775192]
 [ 0.84955052  0.18986199 -0.49215479]
 [-0.50711406  0.03704339 -0.86108252]]
Translation of alignment:
[-0.48805122  2.06917337  1.23543264]
[DEBUG][20

/home/slam/Projects/ORB_SLAM3_DRL/PythonAPI/poseEvaluation.py:204: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  baseline_error_df.rename(columns={'RelativeError': 'BaseRelativeError'}, inplace=True)


[DEBUG][2024-02-08 00:27:09,112][file_interface.read_tum_trajectory_file():117]
Loaded 1554 stamps and poses from: ./logs/trajectory.txt
[DEBUG][2024-02-08 00:27:09,183][file_interface.read_tum_trajectory_file():117]
Loaded 24578 stamps and poses from: ./logs/ground_truth.txt
Subtrajectories for alignment: [[0, 656], [656, 892], [892, 1282], [1282, 1440], [1440, 1554]]
Original estimated subtrajectory length: 656
[DEBUG][2024-02-08 00:27:09,201][sync.associate_trajectories():110]
Found 656 of max. 656 possible matching timestamps between...
	first trajectory
and:	second trajectory
..with max. time diff.: 0.05 (s) and time offset: 0.0 (s).
[DEBUG][2024-02-08 00:27:09,202][trajectory.align():202]
Aligning using Umeyama's method... (with scale correction)
[DEBUG][2024-02-08 00:27:09,204][trajectory.align():214]
Rotation of alignment:
[[-0.18329406  0.98268722  0.02700198]
 [ 0.84858498  0.17202823 -0.50030973]
 [-0.49629308 -0.06879033 -0.86542537]]
Translation of alignment:
[-0.43244736 

/home/slam/Projects/ORB_SLAM3_DRL/PythonAPI/poseEvaluation.py:204: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  baseline_error_df.rename(columns={'RelativeError': 'BaseRelativeError'}, inplace=True)


[DEBUG][2024-02-08 00:27:13,028][file_interface.read_tum_trajectory_file():117]
Loaded 3470 stamps and poses from: ./logs/trajectory.txt
[DEBUG][2024-02-08 00:27:13,176][file_interface.read_tum_trajectory_file():117]
Loaded 48189 stamps and poses from: ./logs/ground_truth.txt
Subtrajectories for alignment: [[0, 2108], [2108, 2431], [2431, 2677], [2677, 2698], [2698, 3470]]
Original estimated subtrajectory length: 2108
[DEBUG][2024-02-08 00:27:13,253][sync.associate_trajectories():110]
Found 2108 of max. 2108 possible matching timestamps between...
	first trajectory
and:	second trajectory
..with max. time diff.: 0.05 (s) and time offset: 0.0 (s).
[DEBUG][2024-02-08 00:27:13,254][trajectory.align():202]
Aligning using Umeyama's method... (with scale correction)
[DEBUG][2024-02-08 00:27:13,259][trajectory.align():214]
Rotation of alignment:
[[-0.12311229  0.94903987  0.29011497]
 [ 0.19405285  0.30971722 -0.93081617]
 [-0.97323525 -0.05829727 -0.22229388]]
Translation of alignment:
[ 1.40

...

/home/slam/Projects/ORB_SLAM3_DRL/PythonAPI/poseEvaluation.py:204: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  baseline_error_df.rename(columns={'RelativeError': 'BaseRelativeError'}, inplace=True)


[DEBUG][2024-02-08 00:28:37,582][file_interface.read_tum_trajectory_file():117]
Loaded 1656 stamps and poses from: ./logs/trajectory.txt
[DEBUG][2024-02-08 00:28:37,650][file_interface.read_tum_trajectory_file():117]
Loaded 23417 stamps and poses from: ./logs/ground_truth.txt
Subtrajectories for alignment: [[0, 195], [195, 214], [214, 680], [680, 703], [703, 914], [914, 1107], [1107, 1132], [1132, 1150], [1150, 1169], [1169, 1347], [1347, 1361], [1361, 1572], [1572, 1596], [1596, 1606], [1606, 1623], [1623, 1656]]
Original estimated subtrajectory length: 195
[DEBUG][2024-02-08 00:28:37,662][sync.associate_trajectories():110]
Found 195 of max. 195 possible matching timestamps between...
	first trajectory
and:	second trajectory
..with max. time diff.: 0.05 (s) and time offset: 0.0 (s).
[DEBUG][2024-02-08 00:28:37,663][trajectory.align():202]
Aligning using Umeyama's method... (with scale correction)
[DEBUG][2024-02-08 00:28:37,664][trajectory.align():214]
Rotation of alignment:
[[-1.8425

subtrajectory alignment failed
Original estimated subtrajectory length: 10
[DEBUG][2024-02-08 00:28:37,740][sync.associate_trajectories():110]
Found 10 of max. 10 possible matching timestamps between...
	first trajectory
and:	second trajectory
..with max. time diff.: 0.05 (s) and time offset: 0.0 (s).
[DEBUG][2024-02-08 00:28:37,740][trajectory.align():202]
Aligning using Umeyama's method... (with scale correction)
[DEBUG][2024-02-08 00:28:37,741][trajectory.align():214]
Rotation of alignment:
[[ 0.09605637 -0.68067389 -0.72626181]
 [ 0.72731732 -0.45012728  0.51806848]
 [-0.67954594 -0.57798658  0.45182832]]
Translation of alignment:
[1.06180689 0.26729295 0.26323194]
[DEBUG][2024-02-08 00:28:37,741][trajectory.align():216]
Scale correction: 1.2007821461709407
Original estimated subtrajectory length: 17
[DEBUG][2024-02-08 00:28:37,743][sync.associate_trajectories():110]
Found 17 of max. 17 possible matching timestamps between...
	first trajectory
and:	second trajectory
..with max. tim

[DEBUG][2024-02-08 00:28:41,548][trajectory.align():216]
Scale correction: 1.1252400883015767
Original estimated subtrajectory length: 34
[DEBUG][2024-02-08 00:28:41,553][sync.associate_trajectories():110]
Found 34 of max. 34 possible matching timestamps between...
	first trajectory
and:	second trajectory
..with max. time diff.: 0.05 (s) and time offset: 0.0 (s).
[DEBUG][2024-02-08 00:28:41,553][trajectory.align():202]
Aligning using Umeyama's method... (with scale correction)
[DEBUG][2024-02-08 00:28:41,554][trajectory.align():214]
Rotation of alignment:
[[-0.17250941  0.98004074 -0.09879605]
 [ 0.85469875  0.09907637 -0.5095821 ]
 [-0.48962286 -0.17234857 -0.8547312 ]]
Translation of alignment:
[0.58538416 0.71440846 1.26153583]
[DEBUG][2024-02-08 00:28:41,554][trajectory.align():216]
Scale correction: 1.2217985705500218
Original estimated subtrajectory length: 11
[DEBUG][2024-02-08 00:28:41,556][sync.associate_trajectories():110]
Found 11 of max. 11 possible matching timestamps betw

IndexError: index 1 is out of bounds for axis 0 with size 1